# 03 — Detección con Mask R-CNN
**Tarea:** Detección de lesiones mamarias — localización mediante bounding boxes + clase (Benigno/Maligno)  
**Arquitectura:** Mask R-CNN con backbone ResNet50-FPN preentrenado en COCO  
**Estrategia:** K-Fold (k=5) sobre Train+Val + Data Augmentation en tiempo real + Evaluación final en Test  

> Las imágenes con lesión (269) son las únicas con anotación de bbox.  
> Las imágenes NORM se excluyen de detección ya que no tienen instancias que localizar.

## 1. Imports y configuración

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

import cv2
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedKFold

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')

## 2. Rutas y parámetros

In [ ]:
DATA_DIR    = Path('../data')
MODELS_DIR  = Path('../models')
RESULTS_DIR = Path('../results')
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE    = 512      # resolución del preprocesamiento
BATCH_SIZE  = 2        # Mask R-CNN consume mucha VRAM; 2 es seguro con 512x512
N_FOLDS     = 5
N_EPOCHS    = 20       # por fold; early stopping lo acortará si corresponde
LR          = 5e-4
PATIENCE    = 5
IOU_THRESH  = 0.5      # umbral IoU para considerar una detección como TP
SCORE_THRESH = 0.5     # umbral de confianza mínimo en inferencia

# Mask R-CNN usa índices de clase empezando en 1 (0 = fondo)
# 1 = Benigno  |  2 = Maligno
NUM_CLASSES = 3        # fondo + Benigno + Maligno
CLASS_NAMES = ['__background__', 'Benigno', 'Maligno']
LABEL_MAP   = {'B': 1, 'M': 2}   # del CSV al índice de clase

## 3. Carga de datos

Usamos los splits guardados en el preprocesamiento.  
Solo trabajamos con imágenes que tienen al menos una lesión anotada (bbox no vacío).  
Las imágenes NORM se excluyen de detección.

In [ ]:
def load_detection_records(split: str) -> pd.DataFrame:
    """
    Construye DataFrame con image_id, img_path, mask_path, bboxes y labels
    para las imágenes que tienen al menos una lesión (bbox no vacío).
    """
    images_dir = DATA_DIR / split / 'images'
    masks_dir  = DATA_DIR / split / 'masks'
    bboxes_dir = DATA_DIR / split / 'bboxes'
    labels_dir = DATA_DIR / split / 'labels'

    records = []
    for img_path in sorted(images_dir.glob('*.png')):
        img_id     = img_path.stem
        bbox_path  = bboxes_dir / f'{img_id}_bboxes.npy'
        label_path = labels_dir / f'{img_id}_label.npy'

        if not bbox_path.exists():
            continue

        bboxes = np.load(str(bbox_path))
        if len(bboxes) == 0:
            continue   # imagen NORM — sin instancias

        clf_label = int(np.load(str(label_path))) if label_path.exists() else 0

        records.append({
            'image_id':  img_id,
            'img_path':  str(img_path),
            'mask_path': str(masks_dir / f'{img_id}.png'),
            'bbox_path': str(bbox_path),
            'clf_label': clf_label   # 1=B, 2=M (para StratifiedKFold)
        })

    return pd.DataFrame(records)


df_train = load_detection_records('train')
df_val   = load_detection_records('val')
df_test  = load_detection_records('test')

# Pool train+val para K-Fold
df_trainval = pd.concat([df_train, df_val], ignore_index=True)

print(f'Train+Val: {len(df_trainval)} imágenes con lesión')
print(f'Test     : {len(df_test)} imágenes con lesión')

## 4. Carga del CSV para etiquetas por instancia

Cada bbox en la máscara corresponde a una lesión.  
Necesitamos la etiqueta B/M **por bbox**, no solo por imagen.  
La derivamos del CSV: si la imagen tiene múltiples lesiones con distintas clases,  
asignamos la clase de cada componente conectado según el orden del CSV.

In [ ]:
CSV_PATH = Path('../data/metadata_clean.csv')
df_csv   = pd.read_csv(CSV_PATH)

def get_instance_labels(img_id: str) -> list:
    """
    Devuelve lista de etiquetas de clase por lesión para una imagen.
    Orden: mismo orden que los componentes conectados de la máscara
    (de arriba-izquierda a abajo-derecha, por y_min).
    Si hay más componentes que filas en el CSV, se asigna la clase predominante.
    """
    rows = df_csv[df_csv['image_id'] == img_id].dropna(subset=['class'])
    rows = rows[rows['class'].isin(['B', 'M'])]

    if len(rows) == 0:
        return []   # no debería ocurrir si filtramos bien

    labels = [LABEL_MAP[c] for c in rows['class'].values]
    return labels


# Verificación rápida
ejemplo = df_trainval['image_id'].iloc[0]
print(f'Etiquetas de instancia para {ejemplo}: {get_instance_labels(ejemplo)}')

## 5. Dataset para Mask R-CNN

Mask R-CNN espera por cada imagen un diccionario con:
- `boxes`  → tensor [N, 4] en formato `[x_min, y_min, x_max, y_max]`
- `labels` → tensor [N] con índice de clase (1=Benigno, 2=Maligno)
- `masks`  → tensor [N, H, W] con máscara binaria por instancia

El augmentation se aplica en tiempo real usando Albumentations,  
garantizando que imagen, máscaras y bboxes se transformen juntos.

In [ ]:
# Augmentation para entrenamiento — imagen, masks y bboxes transformados juntos
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=10, p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.GaussNoise(var_limit=(5, 20), p=0.2),
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['bbox_labels'], min_visibility=0.3))

# Sin augmentation para validación y test
val_transform = None


class MammographyDetectionDataset(Dataset):
    """
    Dataset para Mask R-CNN.
    Carga imagen y máscara real del preprocesamiento.
    Separa la máscara en instancias individuales por componentes conectados.
    Aplica augmentation opcional con Albumentations.
    """
    def __init__(self, dataframe: pd.DataFrame, augment: bool = False):
        self.df      = dataframe.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        img_id = row['image_id']

        # Cargar imagen (ya preprocesada: CLAHE + 512×512)
        image = cv2.imread(row['img_path'], cv2.IMREAD_GRAYSCALE)
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB).astype(np.float32) / 255.0

        # Cargar máscara global
        mask_global = cv2.imread(row['mask_path'], cv2.IMREAD_GRAYSCALE)

        # Separar en instancias mediante componentes conectados
        binary = (mask_global > 127).astype(np.uint8)
        num_cc, labels_cc, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)

        instance_masks = []
        bboxes_xyxy    = []

        for i in range(1, num_cc):
            m    = (labels_cc == i).astype(np.uint8)
            x    = stats[i, cv2.CC_STAT_LEFT]
            y    = stats[i, cv2.CC_STAT_TOP]
            w    = stats[i, cv2.CC_STAT_WIDTH]
            h    = stats[i, cv2.CC_STAT_HEIGHT]
            instance_masks.append(m)
            bboxes_xyxy.append([x, y, x + w, y + h])

        # Etiquetas por instancia desde el CSV
        csv_labels = get_instance_labels(img_id)

        # Alinear número de etiquetas con número de componentes
        # Si hay más componentes que etiquetas, repetimos la última
        n_inst = len(instance_masks)
        if len(csv_labels) == 0:
            csv_labels = [1] * n_inst   # fallback: Benigno
        while len(csv_labels) < n_inst:
            csv_labels.append(csv_labels[-1])
        instance_labels = csv_labels[:n_inst]

        # Augmentation (imagen + bboxes + máscaras de instancia juntos)
        if self.augment and len(bboxes_xyxy) > 0:
            masks_stack = np.stack(instance_masks, axis=-1)   # [H, W, N]
            aug = train_transform(
                image=image,
                masks=[instance_masks[i] for i in range(n_inst)],
                bboxes=bboxes_xyxy,
                bbox_labels=instance_labels
            )
            image           = aug['image']
            instance_masks  = aug['masks']
            bboxes_xyxy     = [list(b) for b in aug['bboxes']]
            instance_labels = list(aug['bbox_labels'])

        # Construir tensores para Mask R-CNN
        img_tensor = torch.tensor(image).permute(2, 0, 1)   # [3, H, W]

        if len(bboxes_xyxy) == 0:
            # Caso sin instancias tras augmentation (raro pero posible)
            target = {
                'boxes':  torch.zeros((0, 4), dtype=torch.float32),
                'labels': torch.zeros(0,      dtype=torch.int64),
                'masks':  torch.zeros((0, IMG_SIZE, IMG_SIZE), dtype=torch.uint8),
            }
        else:
            boxes_t  = torch.tensor(bboxes_xyxy,     dtype=torch.float32)
            labels_t = torch.tensor(instance_labels, dtype=torch.int64)
            masks_t  = torch.tensor(np.stack(instance_masks), dtype=torch.uint8)  # [N, H, W]
            target   = {'boxes': boxes_t, 'labels': labels_t, 'masks': masks_t}

        return img_tensor, target

## 6. Modelo — Mask R-CNN con Transfer Learning desde COCO

Partimos de Mask R-CNN preentrenado en COCO (80 clases).  
Sustituimos solo las cabezas de clasificación y segmentación  
para adaptarlas a nuestras 3 clases (fondo + Benigno + Maligno).

In [ ]:
def build_maskrcnn(num_classes: int) -> nn.Module:
    """
    Mask R-CNN con backbone ResNet50-FPN preentrenado en COCO.
    Solo se reemplazan las cabezas box y mask para num_classes.
    Todo el backbone y el FPN permanecen con pesos COCO → transfer learning.
    """
    model = maskrcnn_resnet50_fpn(
        weights=MaskRCNN_ResNet50_FPN_Weights.COCO_V1
    )

    # Reemplazar cabeza de clasificación de boxes
    in_features_box = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features_box, num_classes)

    # Reemplazar cabeza de segmentación de máscaras
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer     = 256
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, hidden_layer, num_classes)

    return model


# Verificación rápida
_m = build_maskrcnn(NUM_CLASSES)
trainable = sum(p.numel() for p in _m.parameters() if p.requires_grad)
total     = sum(p.numel() for p in _m.parameters())
print(f'Parámetros totales    : {total:,}')
print(f'Parámetros entrenables: {trainable:,}')
del _m

## 7. Función collate — necesaria para Mask R-CNN

Mask R-CNN recibe una lista de `(imagen, target)` por batch, no tensores apilados.  
La función `collate_fn` personalizada evita el comportamiento por defecto de PyTorch.

In [ ]:
def collate_fn(batch):
    return tuple(zip(*batch))


def make_loaders(df_tr: pd.DataFrame, df_vl: pd.DataFrame):
    tr_ds = MammographyDetectionDataset(df_tr, augment=True)
    vl_ds = MammographyDetectionDataset(df_vl, augment=False)
    tr_ld = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,
                       collate_fn=collate_fn, num_workers=2, pin_memory=True)
    vl_ld = DataLoader(vl_ds, batch_size=BATCH_SIZE, shuffle=False,
                       collate_fn=collate_fn, num_workers=2, pin_memory=True)
    return tr_ld, vl_ld

## 8. Métricas de detección — IoU y mAP

Para detección usamos:
- **IoU (Intersection over Union):** qué tanto se solapa el bbox predicho con el real.
- **Precision / Recall / F1** a un umbral de IoU (0.5 = estándar PASCAL VOC).
- **mAP@0.5:** mean Average Precision, métrica principal en detección de objetos.

In [ ]:
def compute_iou(box_a: np.ndarray, box_b: np.ndarray) -> float:
    """IoU entre dos bboxes [x1,y1,x2,y2]."""
    x1 = max(box_a[0], box_b[0]); y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2]); y2 = min(box_a[3], box_b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area_a = (box_a[2]-box_a[0]) * (box_a[3]-box_a[1])
    area_b = (box_b[2]-box_b[0]) * (box_b[3]-box_b[1])
    union  = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def evaluate_detections(predictions: list, targets: list,
                         iou_thresh: float = IOU_THRESH,
                         score_thresh: float = SCORE_THRESH) -> dict:
    """
    Calcula TP, FP, FN, Precision, Recall y F1 a nivel de imagen
    para todas las clases combinadas.

    predictions: lista de dicts {'boxes', 'labels', 'scores'}
    targets    : lista de dicts {'boxes', 'labels'}
    """
    total_tp = total_fp = total_fn = 0

    for pred, tgt in zip(predictions, targets):
        pred_boxes  = pred['boxes'].cpu().numpy()
        pred_scores = pred['scores'].cpu().numpy()
        gt_boxes    = tgt['boxes'].cpu().numpy()

        # Filtrar por score
        keep        = pred_scores >= score_thresh
        pred_boxes  = pred_boxes[keep]

        matched_gt = set()
        tp = fp = 0

        for pb in pred_boxes:
            best_iou, best_j = 0.0, -1
            for j, gb in enumerate(gt_boxes):
                if j in matched_gt:
                    continue
                iou = compute_iou(pb, gb)
                if iou > best_iou:
                    best_iou, best_j = iou, j
            if best_iou >= iou_thresh and best_j >= 0:
                tp += 1
                matched_gt.add(best_j)
            else:
                fp += 1

        fn = len(gt_boxes) - len(matched_gt)
        total_tp += tp; total_fp += fp; total_fn += fn

    precision = total_tp / (total_tp + total_fp + 1e-8)
    recall    = total_tp / (total_tp + total_fn + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)

    return {'precision': precision, 'recall': recall, 'f1': f1,
            'tp': total_tp, 'fp': total_fp, 'fn': total_fn}

## 9. Funciones de entrenamiento y evaluación

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    """
    Mask R-CNN en modo train devuelve directamente un dict de losses.
    Loss total = sum de todas las sub-losses (rpn + box + mask).
    """
    model.train()
    total_loss = 0.0
    n_batches  = 0

    for images, targets in tqdm(loader, desc='  train', leave=False):
        images  = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        optimizer.zero_grad()
        loss_dict = model(images, targets)
        loss      = sum(loss_dict.values())
        loss.backward()
        # Gradient clipping — estabiliza entrenamiento con imágenes médicas
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item()
        n_batches  += 1

    return total_loss / max(n_batches, 1)


@torch.no_grad()
def evaluate_epoch(model, loader, device):
    """
    En modo eval, Mask R-CNN devuelve predicciones (no losses).
    Calculamos métricas de detección sobre el conjunto de validación.
    """
    model.eval()
    all_preds, all_targets = [], []

    for images, targets in loader:
        images = [img.to(device) for img in images]
        preds  = model(images)
        all_preds.extend([{k: v.cpu() for k, v in p.items()} for p in preds])
        all_targets.extend([{k: v.cpu() for k, v in t.items()} for t in targets])

    metrics = evaluate_detections(all_preds, all_targets)
    return metrics, all_preds, all_targets


class EarlyStopping:
    def __init__(self, patience: int, path: str, mode: str = 'max'):
        self.patience = patience; self.path = path; self.mode = mode
        self.best = -np.inf if mode == 'max' else np.inf
        self.counter = 0; self.stop = False

    def __call__(self, metric, model):
        improved = (self.mode == 'max' and metric > self.best) or \
                   (self.mode == 'min' and metric < self.best)
        if improved:
            self.best = metric
            torch.save(model.state_dict(), self.path)
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True

## 10. K-Fold Cross Validation (k=5)

Estratificamos por `clf_label` (B/M) para mantener proporción de clases en cada fold.  
El early stopping monitoriza el F1 de validación (mejor métrica para datasets desbalanceados).

In [ ]:
skf          = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
X            = df_trainval['image_id'].values
y_strat      = df_trainval['clf_label'].values

fold_results   = []
fold_histories = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_strat), start=1):
    print(f'\n{"="*55}')
    print(f'  FOLD {fold}/{N_FOLDS}  |  train: {len(train_idx)}  val: {len(val_idx)}')
    print(f'{"="*55}')

    df_tr = df_trainval.iloc[train_idx].reset_index(drop=True)
    df_vl = df_trainval.iloc[val_idx].reset_index(drop=True)
    train_loader, val_loader = make_loaders(df_tr, df_vl)

    model     = build_maskrcnn(NUM_CLASSES).to(DEVICE)
    optimizer = torch.optim.SGD(
        model.parameters(), lr=LR, momentum=0.9, weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=8, gamma=0.5)
    early_stop = EarlyStopping(
        patience=PATIENCE, path=str(MODELS_DIR / f'maskrcnn_fold{fold}.pt'), mode='max'
    )

    history = {'train_loss': [], 'val_f1': [], 'val_precision': [], 'val_recall': []}

    for epoch in range(1, N_EPOCHS + 1):
        tr_loss = train_one_epoch(model, train_loader, optimizer, DEVICE)
        metrics, _, _ = evaluate_epoch(model, val_loader, DEVICE)
        scheduler.step()
        early_stop(metrics['f1'], model)

        history['train_loss'].append(tr_loss)
        history['val_f1'].append(metrics['f1'])
        history['val_precision'].append(metrics['precision'])
        history['val_recall'].append(metrics['recall'])

        print(f'  Epoch {epoch:3d}/{N_EPOCHS} | '
              f'Loss: {tr_loss:.4f} | '
              f'P: {metrics["precision"]:.4f} '
              f'R: {metrics["recall"]:.4f} '
              f'F1: {metrics["f1"]:.4f}')

        if early_stop.stop:
            print(f'  Early stopping en época {epoch}'); break

    # Recuperar el mejor checkpoint del fold
    model.load_state_dict(torch.load(
        MODELS_DIR / f'maskrcnn_fold{fold}.pt', map_location=DEVICE
    ))
    metrics_best, _, _ = evaluate_epoch(model, val_loader, DEVICE)

    fold_results.append({
        'fold':      fold,
        'val_f1':    metrics_best['f1'],
        'precision': metrics_best['precision'],
        'recall':    metrics_best['recall']
    })
    fold_histories.append(history)
    print(f'  Fold {fold} → F1: {metrics_best["f1"]:.4f} | '
          f'P: {metrics_best["precision"]:.4f} | R: {metrics_best["recall"]:.4f}')


print('\nResumen K-Fold:')
df_kfold = pd.DataFrame(fold_results)
print(df_kfold.to_string(index=False))
print(f"\nMedia F1   : {df_kfold['val_f1'].mean():.4f} ± {df_kfold['val_f1'].std():.4f}")
print(f"Media Prec : {df_kfold['precision'].mean():.4f} ± {df_kfold['precision'].std():.4f}")
print(f"Media Rec  : {df_kfold['recall'].mean():.4f} ± {df_kfold['recall'].std():.4f}")

## 11. Curvas de entrenamiento — todos los folds

In [ ]:
fig, axes = plt.subplots(N_FOLDS, 2, figsize=(14, N_FOLDS * 3))

for i, history in enumerate(fold_histories):
    epochs = range(1, len(history['train_loss']) + 1)

    axes[i, 0].plot(epochs, history['train_loss'], color='steelblue')
    axes[i, 0].set_title(f'Fold {i+1} — Train Loss', fontsize=10)
    axes[i, 0].set_xlabel('Época'); axes[i, 0].grid(alpha=0.3)

    axes[i, 1].plot(epochs, history['val_f1'],        label='F1',        color='tomato')
    axes[i, 1].plot(epochs, history['val_precision'],  label='Precision', color='seagreen', linestyle='--')
    axes[i, 1].plot(epochs, history['val_recall'],     label='Recall',    color='orange',   linestyle=':')
    axes[i, 1].set_title(f'Fold {i+1} — Val Metrics', fontsize=10)
    axes[i, 1].set_xlabel('Época'); axes[i, 1].legend(fontsize=8); axes[i, 1].grid(alpha=0.3)

plt.suptitle('Curvas de entrenamiento Mask R-CNN por fold', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 12. Selección del mejor fold y evaluación en Test

In [ ]:
best_fold = int(df_kfold.loc[df_kfold['val_f1'].idxmax(), 'fold'])
print(f'Mejor fold: {best_fold} (F1 val = {df_kfold.loc[df_kfold["fold"]==best_fold, "val_f1"].values[0]:.4f})')

best_model = build_maskrcnn(NUM_CLASSES).to(DEVICE)
best_model.load_state_dict(
    torch.load(MODELS_DIR / f'maskrcnn_fold{best_fold}.pt', map_location=DEVICE)
)

test_ds = MammographyDetectionDataset(df_test, augment=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=collate_fn, num_workers=2, pin_memory=True)

test_metrics, test_preds, test_targets = evaluate_epoch(best_model, test_loader, DEVICE)

print('\n--- Resultados en Test ---')
print(f'Precision : {test_metrics["precision"]:.4f}')
print(f'Recall    : {test_metrics["recall"]:.4f}')
print(f'F1        : {test_metrics["f1"]:.4f}')
print(f'TP: {test_metrics["tp"]}  FP: {test_metrics["fp"]}  FN: {test_metrics["fn"]}')

## 13. Visualización de predicciones en Test

Verde = bboxes reales (ground truth)  
Rojo = bboxes predichos por Mask R-CNN  
El score de confianza se muestra sobre cada bbox predicho.

In [ ]:
def visualize_detections(dataset, predictions, n: int = 4, score_thresh: float = SCORE_THRESH):
    indices = np.random.choice(len(dataset), min(n, len(dataset)), replace=False)
    fig, axes = plt.subplots(1, len(indices), figsize=(5 * len(indices), 5))
    if len(indices) == 1:
        axes = [axes]

    for ax, idx in zip(axes, indices):
        img_tensor, target = dataset[idx]
        pred = predictions[idx]

        # Imagen en escala de grises
        img_np = img_tensor.permute(1, 2, 0).numpy()[:, :, 0]
        ax.imshow(img_np, cmap='gray')

        # Ground truth (verde)
        for box in target['boxes'].numpy():
            x1,y1,x2,y2 = box
            rect = patches.Rectangle((x1,y1), x2-x1, y2-y1,
                                      linewidth=2, edgecolor='lime', facecolor='none')
            ax.add_patch(rect)

        # Predicciones (rojo)
        boxes  = pred['boxes'].numpy()
        scores = pred['scores'].numpy()
        labels = pred['labels'].numpy()

        for box, score, lbl in zip(boxes, scores, labels):
            if score < score_thresh:
                continue
            x1,y1,x2,y2 = box
            rect = patches.Rectangle((x1,y1), x2-x1, y2-y1,
                                      linewidth=2, edgecolor='red', facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1 - 4, f'{CLASS_NAMES[lbl]} {score:.2f}',
                    color='red', fontsize=7, backgroundcolor='white')

        img_id = dataset.df.iloc[idx]['image_id']
        ax.set_title(img_id, fontsize=9)
        ax.axis('off')

    plt.suptitle('Verde = GT  |  Rojo = Predicción', fontsize=12)
    plt.tight_layout()
    plt.show()


visualize_detections(test_ds, test_preds, n=4)

## 14. Análisis de IoU por imagen

In [ ]:
def compute_mean_iou_per_image(predictions, targets, score_thresh=SCORE_THRESH):
    """Calcula el IoU máximo entre cada GT bbox y las predicciones."""
    ious = []
    for pred, tgt in zip(predictions, targets):
        pred_boxes = pred['boxes'].numpy()
        pred_scores = pred['scores'].numpy()
        gt_boxes    = tgt['boxes'].numpy()

        keep = pred_scores >= score_thresh
        pred_boxes = pred_boxes[keep]

        for gb in gt_boxes:
            if len(pred_boxes) == 0:
                ious.append(0.0)
            else:
                best = max(compute_iou(pb, gb) for pb in pred_boxes)
                ious.append(best)
    return ious


ious = compute_mean_iou_per_image(test_preds, test_targets)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(ious, bins=20, color='steelblue', edgecolor='white')
axes[0].axvline(IOU_THRESH, color='tomato', linestyle='--', label=f'Umbral IoU={IOU_THRESH}')
axes[0].set_xlabel('IoU'); axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de IoU por lesión (Test)', fontsize=12)
axes[0].legend(); axes[0].grid(alpha=0.3)

thresholds = np.arange(0.1, 1.0, 0.05)
recalls_at_t = [np.mean(np.array(ious) >= t) for t in thresholds]
axes[1].plot(thresholds, recalls_at_t, color='seagreen', lw=2)
axes[1].set_xlabel('Umbral IoU'); axes[1].set_ylabel('Recall')
axes[1].set_title('Recall vs umbral IoU (Test)', fontsize=12)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'IoU medio  : {np.mean(ious):.4f}')
print(f'IoU mediano: {np.median(ious):.4f}')
print(f'Recall@0.5 : {np.mean(np.array(ious) >= 0.5):.4f}')

## 15. Guardado de resultados para evaluación comparativa

In [ ]:
# Guardar métricas de test
pd.DataFrame([{
    'model':     'Mask R-CNN',
    'precision': test_metrics['precision'],
    'recall':    test_metrics['recall'],
    'f1':        test_metrics['f1'],
    'mean_iou':  float(np.mean(ious)),
    'tp':        test_metrics['tp'],
    'fp':        test_metrics['fp'],
    'fn':        test_metrics['fn']
}]).to_csv(RESULTS_DIR / 'det_test_metrics.csv', index=False)

# Guardar resumen K-Fold
df_kfold.to_csv(RESULTS_DIR / 'det_kfold_summary.csv', index=False)

# Guardar IoU por lesión
np.save(RESULTS_DIR / 'det_test_ious.npy', np.array(ious))

print('Guardado en', RESULTS_DIR)
print('  det_test_metrics.csv')
print('  det_kfold_summary.csv')
print('  det_test_ious.npy')